Cell 1 — Install dependencies

In [ ]:
!pip install transformers datasets torch scikit-learn pandas requests flask pyngrok -q
print("✅ Dependencies terinstall")

Cell 2 — Konfigurasi

In [ ]:
import os, uuid

# ── ISI BAGIAN INI ─────────────────────────────────────────────
BACKEND_URL = "https://your-backend.com/api"  # ganti dengan URL backend kamu
COLAB_API_KEY = "your-secret-colab-api-key"   # sama dengan COLAB_API_KEY di .env
NGROK_AUTH_TOKEN = "your-ngrok-token"          # dari https://dashboard.ngrok.com
# ──────────────────────────────────────────────────────────────

COLAB_PORT = 5001
WORK_DIR = "/content/transmotion"
SESSION_ID = str(uuid.uuid4())[:8]  # ID unik per sesi Colab

os.makedirs(WORK_DIR, exist_ok=True)

# Header untuk panggilan ke backend
BACKEND_HEADERS = {
    "X-Colab-Key": COLAB_API_KEY,
    "Content-Type": "application/json",
}

print(f"✅ Konfigurasi selesai")
print(f"   Backend    : {BACKEND_URL}")
print(f"   Session ID : {SESSION_ID}")
print(f"   Work dir   : {WORK_DIR}")

Cell 3 — Helper functions

In [ ]:
import json
import math
import os
import random
import time
import traceback
from collections import Counter

import numpy as np
import pandas as pd
import requests
import torch
from datasets import Dataset as HFDataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)


# ── HTTP helpers ───────────────────────────────────────────────

def backend_get(path, params=None):
    try:
        res = requests.get(
            f"{BACKEND_URL}{path}",
            headers={k: v for k, v in BACKEND_HEADERS.items() if k != "Content-Type"},
            params=params,
            timeout=30,
        )
        res.raise_for_status()
        return res.json()
    except Exception as e:
        print(f"❌ GET {path}: {e}")
        return None


def backend_post(path, json_data=None, form_data=None, files=None):
    headers = {k: v for k, v in BACKEND_HEADERS.items() if k != "Content-Type"}
    try:
        if files:
            res = requests.post(
                f"{BACKEND_URL}{path}",
                headers=headers,
                data=form_data or {},
                files=files,
                timeout=120,
            )
        else:
            headers["Content-Type"] = "application/json"
            res = requests.post(
                f"{BACKEND_URL}{path}",
                headers=headers,
                json=json_data or {},
                timeout=30,
            )
        res.raise_for_status()
        return res.json()
    except Exception as e:
        print(f"❌ POST {path}: {e}")
        return None


# ── Reproducibility ────────────────────────────────────────────

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ── Dataset ────────────────────────────────────────────────────

def load_dataset_from_backend(job):
    dataset_id = job["dataset_id"]
    print(f"📦 Mengambil dataset: {job.get('dataset_name', dataset_id)}")

    all_rows, page = [], 1
    while True:
        res = backend_get(
            f"/datasets/{dataset_id}/preprocessed",
            params={"page": page, "per_page": 500},
        )
        if not res or not res.get("data"):
            break
        all_rows.extend(res["data"])
        total_pages = res.get("meta", {}).get("pagination", {}).get("total_pages", 1)
        print(f"   Halaman {page}/{total_pages} — {len(all_rows)} baris...", end="\r")
        if page >= total_pages:
            break
        page += 1

    print(f"\n✅ {len(all_rows)} baris diambil")
    if not all_rows:
        raise ValueError("Dataset kosong")

    df = pd.DataFrame(all_rows)[["preprocessed_text", "label"]].dropna()
    df = df[df["preprocessed_text"].str.strip() != ""].reset_index(drop=True)

    print("   Distribusi kelas:")
    for lbl, cnt in df["label"].value_counts().items():
        print(f"   - {lbl}: {cnt} ({cnt/len(df)*100:.1f}%)")

    return df


def stratified_split(df, test_size, seed=42):
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(
        df, test_size=test_size, stratify=df["label"], random_state=seed
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    print(f"\n📊 Split: train={len(train_df)}, test={len(test_df)}")
    return train_df, test_df


def build_label_maps(labels):
    unique = sorted(set(labels))
    label2id = {l: i for i, l in enumerate(unique)}
    id2label = {i: l for l, i in label2id.items()}
    return label2id, id2label


def tokenize(df, tokenizer, label2id, max_length):
    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
    hf = HFDataset.from_dict({
        "text": df["preprocessed_text"].tolist(),
        "labels": [label2id[l] for l in df["label"].tolist()],
    })
    return hf.map(tokenize_fn, batched=True, remove_columns=["text"], desc="Tokenisasi")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1": float(f1_score(labels, preds, average="weighted", zero_division=0)),
    }


# ── Progress Callback ──────────────────────────────────────────

class BackendProgressCallback(TrainerCallback):
    def __init__(self, job_id, total_epochs):
        self.job_id = job_id
        self.total_epochs = total_epochs

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        progress = int((epoch / self.total_epochs) * 100)
        log = state.log_history[-1] if state.log_history else {}

        backend_post(
            f"/colab/jobs/{self.job_id}/progress",
            json_data={
                "current_epoch": epoch,
                "total_epochs": self.total_epochs,
                "progress": progress,
                "train_loss": round(float(v), 4) if (v := log.get("loss")) else None,
                "val_loss": round(float(v), 4) if (v := log.get("eval_loss")) else None,
                "val_accuracy": round(float(v), 4) if (v := log.get("eval_accuracy")) else None,
                "val_f1": round(float(v), 4) if (v := log.get("eval_f1")) else None,
                "colab_session_id": SESSION_ID,
            },
        )
        print(
            f"\n📡 Epoch {epoch}/{self.total_epochs} — "
            f"loss={log.get('loss', '?'):.4f} "
            f"val_acc={log.get('eval_accuracy', '?')}"
        )


# ── Training utama ─────────────────────────────────────────────

MODEL_BASE_NAMES = {
    "mbert": "bert-base-multilingual-cased",
    "xlmr": "xlm-roberta-base",
}


def train_model(job):
    set_seed(42)

    job_id = job["id"]
    model_type = job["model_type"]
    hp = job["hyperparams"]
    split_info = job.get("split_info", {})

    epochs = hp.get("epochs", 3)
    batch_size = hp.get("batch_size", 16)
    max_length = hp.get("max_length", 128)
    lr = hp.get("learning_rate", 2e-5)
    warmup_steps = hp.get("warmup_steps", 0)
    weight_decay = hp.get("weight_decay", 0.01)
    test_size = split_info.get("test_size", 0.2)
    base_model = MODEL_BASE_NAMES.get(model_type, "bert-base-multilingual-cased")

    print("=" * 55)
    print(f"🚀 Training Job: {job_id[:8]}")
    print(f"   Model: {base_model}")
    print(f"   Epochs: {epochs} | Batch: {batch_size} | LR: {lr}")
    print("=" * 55)

    # 1. Load dataset
    df = load_dataset_from_backend(job)
    train_df, test_df = stratified_split(df, test_size)
    label2id, id2label = build_label_maps(df["label"])
    print(f"\n🏷️  {len(label2id)} kelas: {list(label2id.keys())}")

    # 2. Load model
    print(f"\n📥 Memuat model: {base_model}")
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"   Device: {device.upper()}")

    # 3. Tokenisasi
    print("\n🔤 Tokenisasi...")
    train_ds = tokenize(train_df, tokenizer, label2id, max_length)
    eval_ds = tokenize(test_df, tokenizer, label2id, max_length)

    # 4. Training
    output_dir = os.path.join(WORK_DIR, f"job_{job_id[:8]}")
    os.makedirs(output_dir, exist_ok=True)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        warmup_steps=warmup_steps,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=10,
        save_total_limit=2,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
        callbacks=[
            BackendProgressCallback(job_id, epochs),
            EarlyStoppingCallback(early_stopping_patience=2),
        ],
    )

    print("\n🏋️  Training dimulai...")
    trainer.train()

    # 5. Evaluasi akhir
    print("\n📊 Evaluasi akhir...")
    pred_output = trainer.predict(eval_ds)
    y_pred = pred_output.predictions.argmax(axis=-1)
    y_true = test_df["label"].map(label2id).values

    accuracy = float(accuracy_score(y_true, y_pred))
    f1 = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    precision = float(precision_score(y_true, y_pred, average="weighted", zero_division=0))
    recall = float(recall_score(y_true, y_pred, average="weighted", zero_division=0))

    print(f"\n   Accuracy  : {accuracy*100:.2f}%")
    print(f"   F1 Score  : {f1*100:.2f}%")
    print(f"   Precision : {precision*100:.2f}%")
    print(f"   Recall    : {recall*100:.2f}%")
    print("\n" + classification_report(y_true, y_pred, target_names=list(id2label.values()), zero_division=0))

    # 6. Simpan & upload model
    model_path = os.path.join(output_dir, "model_weights.pt")
    print(f"\n💾 Menyimpan model...")
    torch.save(model.state_dict(), model_path)
    size_mb = os.path.getsize(model_path) / 1024 / 1024
    print(f"   Ukuran: {size_mb:.1f} MB")

    print("\n📤 Mengupload ke backend...")
    job_name = job.get("job_name") or f"{model_type.upper()} Model"
    model_name = f"{job_name} — {time.strftime('%Y%m%d_%H%M')}"
    label_map_json = json.dumps({str(i): l for i, l in id2label.items()})

    with open(model_path, "rb") as f:
        result = backend_post(
            f"/colab/jobs/{job_id}/complete",
            form_data={
                "model_name": model_name,
                "accuracy": str(accuracy),
                "f1_score": str(f1),
                "precision": str(precision),
                "recall": str(recall),
                "label_map": label_map_json,
                "base_model_name": base_model,
                "colab_session_id": SESSION_ID,
            },
            files={"model_file": (f"model_{job_id[:8]}.pt", f, "application/octet-stream")},
        )

    if result and result.get("success"):
        print(f"✅ Model berhasil diupload! ID: {result['data']['id']}")
    else:
        print("❌ Upload gagal")

    # Cleanup
    import shutil
    shutil.rmtree(output_dir, ignore_errors=True)

    return {"accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall}


def report_failure(job_id, error_message):
    backend_post(
        f"/colab/jobs/{job_id}/fail",
        json_data={"error_message": str(error_message)[:1000]},
    )


print("✅ Training functions siap")
print(f"   GPU: {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ tidak tersedia'}")

Cell 4 — Flask Server and Ngrok

In [ ]:
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok, conf

# ── Setup ngrok ────────────────────────────────────────────────
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# ── Flask mini-server ──────────────────────────────────────────
colab_app = Flask(__name__)

# Queue untuk menerima job dari backend
job_queue = []
job_queue_lock = threading.Lock()

@colab_app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "session_id": SESSION_ID,
        "gpu": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })


@colab_app.route("/train", methods=["POST"])
def receive_train():
    """
    Endpoint yang dipanggil Flask backend saat ada job baru.
    Validasi key lalu jalankan training di background thread.
    """
    # Validasi request dari backend
    backend_key = request.headers.get("X-Backend-Key", "")
    if backend_key != COLAB_API_KEY:
        return jsonify({"error": "Unauthorized"}), 401

    job = request.get_json()
    if not job:
        return jsonify({"error": "Body tidak valid"}), 400

    job_id = job.get("id")
    if not job_id:
        return jsonify({"error": "job.id diperlukan"}), 400

    # Konfirmasi ke backend bahwa job sudah diterima
    backend_post(
        f"/colab/jobs/{job_id}/running",
        json_data={"colab_session_id": SESSION_ID},
    )

    # Jalankan training di background thread
    def run():
        try:
            train_model(job)
        except Exception as e:
            error_msg = traceback.format_exc()
            print(f"\n❌ Training error: {e}")
            report_failure(job_id, error_msg)

    thread = threading.Thread(target=run, daemon=True)
    thread.start()

    print(f"\n🔔 Job {job_id[:8]} diterima, training dimulai di background")
    return jsonify({"status": "started", "job_id": job_id, "session_id": SESSION_ID})


@colab_app.route("/shutdown", methods=["POST"])
def shutdown():
    """Backend bisa minta Colab shutdown gracefully."""
    backend_key = request.headers.get("X-Backend-Key", "")
    if backend_key != COLAB_API_KEY:
        return jsonify({"error": "Unauthorized"}), 401

    backend_post(
        "/colab/unregister",
        json_data={"session_id": SESSION_ID},
    )
    return jsonify({"status": "shutting down"})


# ── Jalankan Flask dalam thread ────────────────────────────────
flask_thread = threading.Thread(
    target=lambda: colab_app.run(port=COLAB_PORT, debug=False, use_reloader=False),
    daemon=True,
)
flask_thread.start()
time.sleep(2)  # tunggu Flask siap
print(f"✅ Flask server berjalan di port {COLAB_PORT}")

Cell 5 — ngrok Tunnel + Register ke Backend

In [ ]:
# ── Buka tunnel ────────────────────────────────────────────────
tunnel = ngrok.connect(COLAB_PORT)
COLAB_PUBLIC_URL = tunnel.public_url
print(f"\n🌐 Colab URL: {COLAB_PUBLIC_URL}")

# ── Register ke backend ────────────────────────────────────────
print(f"\n📡 Mendaftarkan ke backend: {BACKEND_URL}")
result = backend_post(
    "/colab/register",
    json_data={
        "url": COLAB_PUBLIC_URL,
        "session_id": SESSION_ID,
    },
)

if result and result.get("success"):
    print(f"✅ Berhasil terdaftar!")
    print(f"   URL    : {COLAB_PUBLIC_URL}")
    print(f"   Session: {SESSION_ID}")
else:
    print("❌ Gagal mendaftar ke backend")
    print("   Pastikan BACKEND_URL dan COLAB_API_KEY sudah benar")

Cell 6 — Heartbeat Loop (jalankan dan biarkan berjalan)

In [ ]:
# ── Heartbeat loop ─────────────────────────────────────────────
# Cell ini mengirim ping ke backend setiap 60 detik agar tidak
# dianggap offline. Biarkan cell ini berjalan selama Colab aktif.
# Tekan ■ Stop untuk menghentikan.

HEARTBEAT_INTERVAL = 60  # detik
CONSECUTIVE_FAILS = 0
MAX_FAILS = 5

print("💓 Heartbeat dimulai")
print(f"   Interval : {HEARTBEAT_INTERVAL}s")
print(f"   Session  : {SESSION_ID}")
print(f"   URL      : {COLAB_PUBLIC_URL}")
print("\n   Tekan ■ Stop untuk menghentikan\n")

while True:
    try:
        result = backend_post(
            "/colab/ping",
            json_data={"session_id": SESSION_ID},
        )
        if result:
            CONSECUTIVE_FAILS = 0
            print(
                f"💓 Ping OK [{time.strftime('%H:%M:%S')}]",
                end="\r",
            )
        else:
            CONSECUTIVE_FAILS += 1
            print(f"\n⚠️  Ping gagal ({CONSECUTIVE_FAILS}/{MAX_FAILS})")
            if CONSECUTIVE_FAILS >= MAX_FAILS:
                print("❌ Backend tidak merespons. Coba re-register.")
                # Coba register ulang
                backend_post(
                    "/colab/register",
                    json_data={"url": COLAB_PUBLIC_URL, "session_id": SESSION_ID},
                )
                CONSECUTIVE_FAILS = 0

    except KeyboardInterrupt:
        print("\n\n⏹️  Heartbeat dihentikan")
        backend_post("/colab/unregister", json_data={"session_id": SESSION_ID})
        ngrok.disconnect(COLAB_PUBLIC_URL)
        print("   Colab berhasil unregister dari backend")
        break
    except Exception as e:
        print(f"\n⚠️  Heartbeat error: {e}")

    time.sleep(HEARTBEAT_INTERVAL)